# 04 – Demo-Server auf Colab (API + ngrok) für den iOS-Kurzbefehl

Startet die DoppelCheck-API auf Colab und macht sie per ngrok öffentlich erreichbar.
Die ausgegebene Adresse wird im iOS-Kurzbefehl (Aktion „Inhalte von URL abrufen“) eingetragen – siehe `shortcut/README.md`.

**Vorbereitung**
1. Kostenloses Konto bei https://ngrok.com → *Your Authtoken* kopieren.
2. In Colab links das Schlüssel-Symbol (*Secrets*) → Secret `NGROK_AUTHTOKEN` anlegen und für dieses Notebook freigeben.
3. Optional: Ordner `DoppelCheck/models/distilbert` aus Google Drive nach `models/distilbert/` kopieren (Zelle 3), sonst läuft die Baseline.
4. *Laufzeit → Alle ausführen*. CPU reicht. Die Adresse ändert sich bei jedem Start.

In [ ]:
# --- 1. Repo klonen, Pakete installieren ---------------------------------------------
REPO = "jhyoun2028/BWKI_dp"
BRANCH = "main"            # ggf. auf den Arbeits-Branch setzen
REPO_DIR = "/content/BWKI_dp"

import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO}.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
%pip install -q -r requirements.txt pyngrok

In [ ]:
# --- 2. (optional) feingetuntes Modell von Google Drive holen --------------------------
USE_DRIVE_MODEL = True
if USE_DRIVE_MODEL:
    from google.colab import drive
    drive.mount("/content/drive")
    src = "/content/drive/MyDrive/DoppelCheck/models/distilbert"
    if os.path.isdir(src):
        subprocess.run(["cp", "-r", src, "models/distilbert"], check=True)
        print("DistilBERT aus Drive kopiert → models/distilbert/")
    else:
        print("Kein Modell unter", src, "→ Baseline wird verwendet")

In [ ]:
# --- 3. OCR-Gewichte einmalig laden (dauert ~1 Minute) ---------------------------------
import sys; sys.path.insert(0, "src")
import ocr
print("OCR bereit:", ocr.ocr_available())

In [ ]:
# --- 4. API im Hintergrund starten -----------------------------------------------------
import time, requests
api = subprocess.Popen(["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"],
                       stdout=open("/content/api.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.ok:
            print("API läuft:", r.json()); break
    except Exception:
        time.sleep(2)
else:
    print(open("/content/api.log").read()); raise RuntimeError("API startet nicht")

In [ ]:
# --- 5. Öffentliche Adresse per ngrok ---------------------------------------------------
from google.colab import userdata
from pyngrok import ngrok
ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
tunnel = ngrok.connect(8000, "http")
PUBLIC_URL = tunnel.public_url.replace("http://", "https://")
print("Öffentliche Adresse für den iOS-Kurzbefehl:")
print(f"   {PUBLIC_URL}/scan")
print("Test:", requests.get(f"{PUBLIC_URL}/health", timeout=10).json())

In [ ]:
# --- 6. Probe mit einem Beispiel-Screenshot ---------------------------------------------
with open("data/samples/dhl_phishing.png", "rb") as f:
    r = requests.post(f"{PUBLIC_URL}/scan", files={"file": ("shot.png", f, "image/png")}, timeout=120)
print(r.status_code, r.json())
print("\nServer läuft weiter, solange dieses Notebook offen ist. Zum Beenden: Laufzeit → Laufzeit trennen.")